# 05 - Concurrency I: Threading, the GIL & Multiprocessing

Part of the Python, DSA & Git chapter. This notebook answers the single most commonly asked concurrency question in Python interviews: should threading or multiprocessing be used here, and why. The answer hinges entirely on the Global Interpreter Lock, covered first, then demonstrated with real benchmarks rather than just asserted.

Covers: what the GIL is and why it exists, threading basics, race conditions and locks, ThreadPoolExecutor and ProcessPoolExecutor, multiprocessing basics, and a decision framework for I/O-bound vs CPU-bound work. Practice exercises at the end.

# Part A - The GIL: the Single Most Important Fact About Python Concurrency

## What the GIL is, and why it exists

CPython, the standard Python implementation, has a Global Interpreter Lock: only one thread can execute Python bytecode at a time, no matter how many threads exist or how many CPU cores are available. It exists mainly because CPython uses reference counting for memory management, and reference counts need to be updated atomically -- the GIL is a simple, if blunt, way to guarantee that without fine-grained locks around every single object.

The practical consequence: threads in CPython give concurrency, meaning interleaved work, not parallelism, meaning simultaneous execution across cores, for CPU-bound Python code. The GIL IS released during I/O waits, such as network calls, disk reads, and time.sleep, and during many C-extension operations such as NumPy array math -- which is exactly why threading still helps enormously for I/O-bound work, and why NumPy-heavy code can sometimes see real threaded speedups despite the GIL.

## Proof, not assertion: CPU-bound work does not speed up with threads

In [1]:
import time
import threading

def cpu_bound_task(n):
    count = 0
    for i in range(n):
        count += i * i
    return count

N = 4_000_000

start = time.perf_counter()
for _ in range(4):
    cpu_bound_task(N)
sequential_time = time.perf_counter() - start
print("4x sequential (single thread):", round(sequential_time, 3), "s")

start = time.perf_counter()
threads = [threading.Thread(target=cpu_bound_task, args=(N,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
threaded_time = time.perf_counter() - start
print("4 threads:", round(threaded_time, 3), "s")
print("speedup:", round(sequential_time / threaded_time, 2), "x  <- nowhere near 4x, this is the GIL in action")

4x sequential (single thread): 0.885 s


4 threads: 0.854 s
speedup: 1.04 x  <- nowhere near 4x, this is the GIL in action


## The other half of the proof: I/O-bound work DOES speed up with threads

time.sleep releases the GIL while waiting, which is exactly what happens during a real network call, database query, or disk read -- the thread is not doing Python work, it is waiting, so other threads are free to run.

In [2]:
def io_bound_task(duration):
    time.sleep(duration)          # stands in for a network call, DB query, or disk read
    return duration

start = time.perf_counter()
for _ in range(4):
    io_bound_task(0.25)
sequential_io_time = time.perf_counter() - start
print("4x sequential (single thread):", round(sequential_io_time, 3), "s")

start = time.perf_counter()
threads = [threading.Thread(target=io_bound_task, args=(0.25,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
threaded_io_time = time.perf_counter() - start
print("4 threads:", round(threaded_io_time, 3), "s")
print("speedup:", round(sequential_io_time / threaded_io_time, 2), "x  <- close to 4x, GIL was released during each sleep")

4x sequential (single thread): 1.001 s


4 threads: 0.251 s
speedup: 3.99 x  <- close to 4x, GIL was released during each sleep


# Part B - Threading Fundamentals

## Creating and joining threads

threading.Thread(target=fn, args=...) creates a thread; .start() begins running it; .join() blocks the calling thread until it finishes. Forgetting .join() is a common bug -- the main program can move on before background threads finish their work.

In [3]:
def worker(name, delay):
    time.sleep(delay)
    print(f"worker {name} finished after {delay}s")

threads = [threading.Thread(target=worker, args=(f"T{i}", 0.1 * i)) for i in range(1, 4)]
for t in threads:
    t.start()
print("all threads started, main thread continues immediately")
for t in threads:
    t.join()                       # wait here until every thread has actually finished
print("all threads joined -- guaranteed finished by this point")

all threads started, main thread continues immediately
worker T1 finished after 0.1s


worker T2 finished after 0.2s


worker T3 finished after 0.30000000000000004s
all threads joined -- guaranteed finished by this point


## Race conditions: what goes wrong without synchronization

A race condition happens when multiple threads read and write shared state without coordination, and the final result depends on unpredictable timing. counter += 1 looks like a single operation but is not -- it is really read, add one, write back, and threads can interleave in the middle of that sequence. To make the failure reliably visible in a short demo rather than leaving it to chance timing, the read and write are split apart with an explicit time.sleep(0) between them, which forces a context switch right at the dangerous moment -- in real, unmodified code the exact same failure can happen, just less predictably.

In [4]:
counter = 0

def increment_unsafe(n):
    global counter
    for _ in range(n):
        temp = counter             # READ
        time.sleep(0)               # force a context switch here, for a reliable demo
        counter = temp + 1          # WRITE -- another thread may have changed counter in between

counter = 0
threads = [threading.Thread(target=increment_unsafe, args=(500,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("expected:", 4 * 500)
print("actual:  ", counter, "  <- LESS than expected: lost updates from the race condition")

expected: 2000
actual:   503   <- LESS than expected: lost updates from the race condition


## Fixing it with a Lock

threading.Lock guarantees only one thread can hold it at a time. The `with lock:` form is preferred because it releases the lock automatically even if the block raises, exactly like the context managers from the previous notebook. Wrapping the same read-sleep-write sequence that just broke above in a lock fixes it completely, even under the exact same forced interleaving.

In [5]:
counter = 0
lock = threading.Lock()

def increment_safe(n):
    global counter
    for _ in range(n):
        with lock:                  # only one thread executes this block at a time
            temp = counter
            time.sleep(0)            # the same adversarial interleaving as above...
            counter = temp + 1       # ...but now it cannot cause a lost update

counter = 0
threads = [threading.Thread(target=increment_safe, args=(500,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("expected:", 4 * 500)
print("actual:  ", counter, "  <- now always exactly correct, even under forced interleaving")

expected: 2000
actual:   2000   <- now always exactly correct, even under forced interleaving


## Other synchronization primitives, briefly

- RLock: a Lock the SAME thread can acquire multiple times without deadlocking itself, needed for recursive functions that take a lock.
- Semaphore: allows up to N threads through at once, useful for capping concurrent access to a limited resource such as a connection pool.
- Event: a simple flag threads can wait on, useful for signaling start-now or stop-now across threads.

A deadlock happens when two or more threads each hold a lock the other needs, and neither can proceed -- for example, thread A holds lock 1 and waits for lock 2, while thread B holds lock 2 and waits for lock 1. The standard prevention strategy is simple: always acquire multiple locks in the same, globally agreed order, everywhere in the codebase.

In [6]:
semaphore = threading.Semaphore(2)     # at most 2 threads through at once

def limited_worker(name):
    with semaphore:
        print(f"{name} acquired the semaphore")
        time.sleep(0.2)
        print(f"{name} releasing the semaphore")

threads = [threading.Thread(target=limited_worker, args=(f"W{i}",)) for i in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

W0 acquired the semaphoreW1 acquired the semaphore



W1 releasing the semaphore


W0 releasing the semaphore
W2 acquired the semaphore
W3 acquired the semaphore


W3 releasing the semaphore
W2 releasing the semaphore


# Part C - ThreadPoolExecutor and ProcessPoolExecutor

## ThreadPoolExecutor: the modern, preferred way to manage threads

Manually creating and joining threads gets unwieldy past a handful of them. concurrent.futures.ThreadPoolExecutor manages a pool of worker threads: .map() runs a function across an iterable and returns results in order; .submit() plus as_completed() processes results as they finish, in whatever order that happens to be.

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fake_api_call(request_id):
    time.sleep(0.2)                 # simulate network latency
    return f"response for request {request_id}"

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fake_api_call, range(5)))
print(round(time.perf_counter() - start, 3), "s for 5 calls via .map() -- results stay IN ORDER")
print(results)

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(fake_api_call, i): i for i in range(5)}
    for future in as_completed(futures):          # yields whichever finishes first
        print("completed:", future.result())
print(round(time.perf_counter() - start, 3), "s via submit()/as_completed() -- order NOT guaranteed")

0.202 s for 5 calls via .map() -- results stay IN ORDER
['response for request 0', 'response for request 1', 'response for request 2', 'response for request 3', 'response for request 4']
completed: response for request 0


completed: response for request 2
completed: response for request 3
completed: response for request 1
completed: response for request 4
0.203 s via submit()/as_completed() -- order NOT guaranteed


## ProcessPoolExecutor: real parallelism for CPU-bound work

ProcessPoolExecutor has the identical interface to ThreadPoolExecutor but runs each task in a SEPARATE PROCESS, each with its own Python interpreter and its own GIL, so CPU-bound work actually runs in parallel across cores. The cost: data passed to and returned from worker processes must be pickled to cross process boundaries, overhead that does not exist with threads, since threads share memory directly.

In [8]:
import os
from concurrent.futures import ProcessPoolExecutor

print(f"this machine reports {os.cpu_count()} CPU core(s) available")

N = 4_000_000
start = time.perf_counter()
for _ in range(4):
    cpu_bound_task(N)
sequential_time = time.perf_counter() - start
print("4x sequential:", round(sequential_time, 3), "s")

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(cpu_bound_task, [N] * 4))
process_time = time.perf_counter() - start
print("4 processes:", round(process_time, 3), "s")
print("speedup:", round(sequential_time / process_time, 2), "x")

if os.cpu_count() > 1:
    print()
    print("on multi-core hardware this should approach a real speedup, unlike threading above")
else:
    print()
    print("this environment has only 1 CPU core, so there are no idle cores to parallelize onto --")
    print("the numbers above mainly reflect process-spawning and pickling overhead, not a slowdown")
    print("caused by the GIL. on a multi-core machine, rerun this cell and expect a real speedup.")

this machine reports 1 CPU core(s) available


4x sequential: 0.858 s


4 processes: 0.983 s
speedup: 0.87 x

this environment has only 1 CPU core, so there are no idle cores to parallelize onto --
the numbers above mainly reflect process-spawning and pickling overhead, not a slowdown
caused by the GIL. on a multi-core machine, rerun this cell and expect a real speedup.


## multiprocessing.Pool and a portability note

concurrent.futures.ProcessPoolExecutor is built on top of the lower-level multiprocessing module, which can also be used directly via multiprocessing.Pool. One portability detail worth knowing: on Linux, the default process start method is fork, which is why the cells above work without any special guard. On Windows and macOS, the default start method is spawn, which re-imports the main script in each new process -- without wrapping multiprocessing code in `if __name__ == "__main__":`, spawn can trigger infinite recursion. Production code that needs to run cross-platform should include that guard, even though it was not needed here:

```python
import multiprocessing

def square(x):
    return x * x

if __name__ == "__main__":          # required for spawn on Windows/macOS
    with multiprocessing.Pool(processes=2) as pool:
        print(pool.map(square, range(10)))
```

In [9]:
import multiprocessing

def square(x):
    return x * x

with multiprocessing.Pool(processes=2) as pool:
    results = pool.map(square, range(10))
print(results)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


# Part D - Decision Framework

## Threading vs multiprocessing vs asyncio: the decision that actually matters

| Workload | Use | Why |
|---|---|---|
| I/O-bound: network calls, file/disk I/O, DB queries, waiting on external APIs | threading (ThreadPoolExecutor) or asyncio (next notebook) | GIL is released during I/O waits, so threads genuinely overlap that waiting time |
| CPU-bound: heavy computation, data crunching in pure Python | multiprocessing (ProcessPoolExecutor) | separate processes each get their own GIL, enabling real parallel execution across cores |
| CPU-bound, but using NumPy/pandas/PyTorch | threading often still works | many C-extension operations release the GIL internally during the heavy computation |
| Very high number of concurrent I/O tasks, such as thousands of open connections | asyncio | thread-per-task does not scale to thousands of OS threads; a single-threaded event loop does, covered next |

Interview-ready one-liner: **threading gives concurrency for I/O-bound work despite the GIL; multiprocessing gives real parallelism for CPU-bound work by sidestepping the GIL entirely, at the cost of process-spawning and serialization overhead.**

One more note for completeness: Python 3.13 introduced an experimental free-threaded build that can disable the GIL entirely. It is not yet the default build, and ecosystem support, especially C-extension compatibility, is still maturing -- worth knowing this landscape is actively changing, but not yet the default assumption to interview with.

## Practice exercises

Implement each TODO, then run the check cell.

In [10]:
class SafeCounter:
    # Thread-safe counter: increment() must be safe to call from multiple threads at once.
    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()

    def increment(self):
        # TODO: implement using self._lock so concurrent increments cannot race
        raise NotImplementedError

    @property
    def value(self):
        return self._value


def parallel_map_sum(fn, items, n_workers=4):
    # Apply fn to every item using a ThreadPoolExecutor with n_workers workers,
    # and return the SUM of all the results.
    # TODO: implement using concurrent.futures.ThreadPoolExecutor
    raise NotImplementedError

In [11]:
def _check(label, ok, detail=""):
    print("  [" + ("PASS" if ok else "FAIL") + "]", label, detail)

try:
    sc = SafeCounter()
    thread_errors = []

    def bump():
        try:
            for _ in range(1000):
                sc.increment()
        except Exception as e:
            thread_errors.append(e)

    workers = [threading.Thread(target=bump) for _ in range(8)]
    for t in workers:
        t.start()
    for t in workers:
        t.join()

    if thread_errors and isinstance(thread_errors[0], NotImplementedError):
        print("  [SKIP] SafeCounter -- not implemented yet")
    elif thread_errors:
        print("  [ERROR] SafeCounter --", thread_errors[0])
    else:
        _check("SafeCounter reaches exactly 8000 under concurrent increments", sc.value == 8000, f"(got {sc.value})")
except Exception as e:
    print("  [ERROR] SafeCounter --", e)

try:
    result = parallel_map_sum(lambda x: x * x, [1, 2, 3, 4], n_workers=2)
    _check("parallel_map_sum computes sum of squares", result == 30, f"(got {result})")
except NotImplementedError:
    print("  [SKIP] parallel_map_sum -- not implemented yet")
except Exception as e:
    print("  [ERROR] parallel_map_sum --", e)

  [SKIP] SafeCounter -- not implemented yet
  [SKIP] parallel_map_sum -- not implemented yet


## Self-check before moving on

- [ ] I can explain what the GIL is and why CPython has it
- [ ] I can explain why threading helps I/O-bound work but not CPU-bound work, backed by a benchmark, not just a claim
- [ ] I can identify and fix a race condition using a Lock
- [ ] I know when to reach for Semaphore vs Lock, and what a deadlock is
- [ ] I can use ThreadPoolExecutor with both .map() and submit()/as_completed()
- [ ] I know ProcessPoolExecutor achieves real parallelism because each process has its own GIL, and what that costs in pickling and spawn overhead
- [ ] I can state the one-line decision rule: I/O-bound uses threading or asyncio, CPU-bound uses multiprocessing

Next: `06-async-await-asyncio.ipynb`